## 1 - Configurações Gerais

In [1]:
#@markdown Detectando o Ambiente de Execução { display-mode: "form" }
try:
    import google.colab
    EXECUTION_ENV = "colab"
except ImportError:
    try:
        import kaggle_secrets
        EXECUTION_ENV = "kaggle"
    except ImportError:
        EXECUTION_ENV = "local"

print(f"AMBIENTE DE EXECUÇÃO DETECTADO: {EXECUTION_ENV}")

AMBIENTE DE EXECUÇÃO DETECTADO: local


In [2]:
if EXECUTION_ENV in ["colab", "kaggle"]:
    !pip install -q minigrid==3.1.0
    !pip install -q langchain_openai

In [3]:
import os
import sys

if EXECUTION_ENV == "colab":
    repository_path = os.path.join(os.getcwd(), "minigrid_benchmark")

if EXECUTION_ENV == "kaggle":
    repository_path = os.path.join("/kaggle/working/", "minigrid_benchmark")

if EXECUTION_ENV in ["colab", "kaggle"]:
    if not os.path.exists(repository_path):
        !git clone https://github.com/pablo-sampaio/minigrid_benchmark.git
    repository_src_path = os.path.join(repository_path, "src")
    sys.path.append(repository_src_path)
    print(f"Repository src path: {repository_src_path}")

In [4]:
from langchain_openai import ChatOpenAI

import experiments_util
from experiments_util import run_and_save_experiments, create_experiment_config

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [5]:
if EXECUTION_ENV == "colab":
    # Mount Google Drive, to save results there
    from google.colab import drive
    drive.mount('/content/drive')
    results_dir = '/content/drive/My Drive/EAD-Pesquisa-Agentes/_projeto_minigrid/results'

if EXECUTION_ENV == "kaggle":
    results_dir = "/kaggle/working/results"

if EXECUTION_ENV in ["colab", "kaggle"]:
    if not os.path.exists(results_dir):
        os.makedirs(results_dir)
    experiments_util.RESULTS_BASE_DIR = results_dir

In [6]:
OPENAI_KEY_NAME = "OPENAI_PABLO"
OPENAI_API_KEY = None

if EXECUTION_ENV == "colab":
    # just assert that HUGGINGFACE_API_KEY or KF_TOKEN secret is set
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get(OPENAI_KEY_NAME)

if EXECUTION_ENV == "kaggle":
    # para o Kaggle
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    OPENAI_API_KEY = user_secrets.get_secret(OPENAI_KEY_NAME)

if EXECUTION_ENV == "local":
    OPENAI_API_KEY = os.getenv(OPENAI_KEY_NAME)

if not OPENAI_API_KEY:
    raise ValueError(f"{OPENAI_KEY_NAME} environment variable is not set")

## 2 - Configurar o Modelo

In [ ]:
MODEL_NAME = "gpt-5.4-mini"

In [ ]:
MODEL_GPT_5_4 = ChatOpenAI(
    model=MODEL_NAME,
    api_key=OPENAI_API_KEY,
)

## 3 - Configurações do Experimento

In [ ]:

configs = [
    # experimentos com visao global
    create_experiment_config(MODEL_NAME, MODEL_GPT_5_4, True, False, False, history_size=1),
    create_experiment_config(MODEL_NAME, MODEL_GPT_5_4, True, True, True, history_size=1),
    create_experiment_config(MODEL_NAME, MODEL_GPT_5_4, True, False, False, history_size=5),
    create_experiment_config(MODEL_NAME, MODEL_GPT_5_4, True, True, True, history_size=5),

    # experimentos com visao local (só com gpt-5.4)
    create_experiment_config(MODEL_NAME, MODEL_GPT_5_4, False, False, False, history_size=1),
    create_experiment_config(MODEL_NAME, MODEL_GPT_5_4, False, True, True, history_size=1),
    create_experiment_config(MODEL_NAME, MODEL_GPT_5_4, False, False, False, history_size=5),
    create_experiment_config(MODEL_NAME, MODEL_GPT_5_4, False, True, True, history_size=5),
];

In [ ]:
'''
configs.extend([
    # experimento de historico (para comparar 1, 3, 5 e 9)
    create_experiment_config(MODEL_NAME, MODEL_GPT, True, False, False, history_size=3),
    create_experiment_config(MODEL_NAME, MODEL_GPT, True, True, True, history_size=3),
    create_experiment_config(MODEL_NAME, MODEL_GPT, True, False, False, history_size=9),
    create_experiment_config(MODEL_NAME, MODEL_GPT, True, True, True, history_size=9),
])
'''

In [ ]:
f"{len(configs)} configurations to run!"

## 4 - Execução do Experimento

In [ ]:
experiment_name = "2026-06-02_openai_{mode}"   # se já existir, continua de onde parou

In [24]:
##########################
##  Execute experiments ##
##########################

final_results, filepath = run_and_save_experiments(configs, experiment_name=experiment_name, verbose=True)


Resuming from: d:\Pablo\Documents\Projects\pablo-sampaio\minigrid_benchmark\results\2026-06-01b_openai\summary.json
Results will be saved to: d:\Pablo\Documents\Projects\pablo-sampaio\minigrid_benchmark\results\2026-06-01b_openai\summary.json


Completed Experiment Configurations:   0%|          | 0/2 [00:00<?, ?it/s]

Completed Environments:   0%|          | 0/2 [00:00<?, ?it/s]

Finished Environment Runs::   0%|          | 0/5 [00:00<?, ?it/s]

Finished Environment Runs::   0%|          | 0/5 [00:00<?, ?it/s]

Completed Environments:   0%|          | 0/2 [00:00<?, ?it/s]

Finished Environment Runs::   0%|          | 0/5 [00:00<?, ?it/s]

Finished Environment Runs::   0%|          | 0/5 [00:00<?, ?it/s]

In [25]:
if EXECUTION_ENV in ["colab", "kaggle"]:
    # zip the final results directory
    import shutil

    experiment_result_dir = os.path.dirname(filepath)
    zip_path = os.path.join(experiments_util.RESULTS_BASE_DIR, f"{experiment_name}_results_zip")

    # Creates results_zip.zip containing everything
    shutil.make_archive(zip_path, 'zip', experiment_result_dir)

    print(f"Created: {zip_path}.zip")

In [ ]:
print("Saved results to:", filepath)
print("Now, run the analysis notebook.")

Saved results to: d:\Pablo\Documents\Projects\pablo-sampaio\minigrid_benchmark\results\2026-06-01b_openai\summary.json
Now, run notebook "run_experiments_analysis.ipynb".
